In [32]:
# MobileNetV2 extracts visual features from the image.
# The Dense layer receives those features and predicts probabilities for the four classes.
# The loss function measures how far the prediction is from the correct label.
# Backpropagation updates the trainable classification-head weights.
# The frozen MobileNetV2 weights stay unchanged.

In [42]:
from tensorflow.keras.applications import MobileNetV2

In [43]:
OUTPUT_DIR = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [44]:
# Model Architecture

def create_tumor_classifier():

    # define the input layer
    inputs = layers.Input(
        shape = (224,224,3),
        name = "mri_images"
    )
    
    # rescaling the images to -1 , 1 for MobileNetV2 (we can use process_input())
    # x new = 2 * x old - 1
    x = layers.Rescaling(
        scale = 2.0,
        offset = -1.0,
        name = 'mobilenet_preprocessing'
    )(inputs)

    #loading the pretrained MobileNetV2
    base_model = MobileNetV2( weights ='imagenet', include_top = False, input_shape =(224, 224, 3))

    # freeze the pretrained layers
    base_model.trainable = False

    # extract images features
    x = base_model(x, training=False)


    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    
    outputs = layers.Dense(4, activation='softmax', name='Classification')(x)

    # Assemble the model

    model = Model(
        inputs = inputs,
        outputs = outputs
    )

    return model, base_model




In [47]:
model, base_model = create_tumor_classifier()

model.summary()

# base_model.summary()



Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mri_images (InputLayer)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_preprocessing         │ (None, 224, 224, 3)    │             0 │
│ (Rescaling)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Classification (Dense)          │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [39]:
# compile the model 

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate =0.001 ), # Adam: updates the trainable weights.
    loss= 'categorical_crossentropy',
    metrics = ['accuracy']
)


In [42]:
model_checkpoint